# 📈 Notebook 05 — Evaluation & Results

**CO543/CO5430 — Traffic Sign Detection**

**Goal**: Compare all model variants on the held-out **test split** and produce all figures and tables for the final report.

**M6 Deliverable**:
- ✅ Full results table (all model variants)
- ✅ Precision-Recall curves
- ✅ Qualitative success & failure examples
- ✅ Failure case analysis

> ⚠️ **Test set discipline**: Only run this notebook once per model variant. Never tune hyperparameters based on test set results.

In [ ]:
import sys, json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.classical_detector import ClassicalDetector
from src.utils.metrics import compute_iou, compute_precision_recall, evaluate_classical_baseline
from src.utils.visualization import plot_pr_curve, plot_confusion_matrix, save_qualitative_grid

TEST_DIR    = PROJECT_ROOT / 'data' / 'processed' / 'gtsdb' / 'test'
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'gtsdb_yolov8n.yaml'
RESULTS_DIR = PROJECT_ROOT / 'results'
CLASS_NAMES = {0: 'Prohibitory', 1: 'Danger', 2: 'Mandatory'}

print(f'Test directory: {TEST_DIR}')
print(f'Exists: {TEST_DIR.exists()}')

## 1. Load Existing Metrics (from previous notebooks)

In [ ]:
def load_metrics(json_path: Path) -> dict:
    if json_path.exists():
        with open(json_path) as f:
            return json.load(f)
    return {}

metrics_dir = RESULTS_DIR / 'metrics'
classical   = load_metrics(metrics_dir / 'classical_baseline.json')
zero_shot   = load_metrics(metrics_dir / 'zero_shot_baseline.json')

print('Classical baseline:', classical)
print('Zero-shot baseline:', zero_shot)

## 2. Evaluate Fine-Tuned Models on Test Set

> ⚠️ Run this only ONCE per model variant on the test set.

In [ ]:
# Define checkpoints to evaluate
CHECKPOINTS = {
    'YOLOv8n (fine-tuned, with aug)':    PROJECT_ROOT / 'results' / 'checkpoints' / 'gtsdb_yolov8n_v1_best.pt',
    'YOLOv8n (fine-tuned, no aug)':      PROJECT_ROOT / 'results' / 'checkpoints' / 'gtsdb_yolov8n_noaug_v1_best.pt',
    'YOLOv8s (fine-tuned, with aug)':    PROJECT_ROOT / 'results' / 'checkpoints' / 'gtsdb_yolov8s_v1_best.pt',
}

fine_tuned_metrics = {}

for model_name, ckpt_path in CHECKPOINTS.items():
    if not ckpt_path.exists():
        print(f'[SKIP] {model_name}: checkpoint not found at {ckpt_path}')
        continue

    print(f'\nEvaluating: {model_name}')
    model = YOLO(str(ckpt_path))
    val_results = model.val(data=str(CONFIG_PATH), split='test', verbose=False)

    m = {
        'map50':     round(float(val_results.box.map50), 4),
        'map50_95':  round(float(val_results.box.map),   4),
        'precision': round(float(val_results.box.mp),    4),
        'recall':    round(float(val_results.box.mr),    4),
    }
    fine_tuned_metrics[model_name] = m
    print(f'  mAP@0.5      : {m["map50"]}')
    print(f'  mAP@0.5:0.95 : {m["map50_95"]}')
    print(f'  Precision    : {m["precision"]}')
    print(f'  Recall       : {m["recall"]}')

    out_path = metrics_dir / f'{model_name.lower().replace(" ","_").replace("(","").replace(")","").replace(",","")}.json'
    with open(out_path, 'w') as f:
        json.dump({'model': model_name, **m}, f, indent=2)
    print(f'  Saved → {out_path}')

## 3. Full Results Table (All Models)

In [ ]:
rows = []

if classical:
    rows.append({'Model': 'Classical CV Baseline (HSV+Contour)',
                 'mAP@0.5': '—',
                 'mAP@0.5:0.95': '—',
                 'Precision': f"{classical.get('precision', 0):.4f}",
                 'Recall':    f"{classical.get('recall',    0):.4f}",
                 'AP@0.5':   f"{classical.get('ap',        0):.4f}"})

if zero_shot:
    rows.append({'Model': 'Zero-Shot YOLOv8n (COCO, no fine-tune)',
                 'mAP@0.5': '—',
                 'mAP@0.5:0.95': '—',
                 'Precision': f"{zero_shot.get('precision', 0):.4f}",
                 'Recall':    f"{zero_shot.get('recall',    0):.4f}",
                 'AP@0.5':   f"{zero_shot.get('ap',        0):.4f}"})

for model_name, m in fine_tuned_metrics.items():
    rows.append({'Model': model_name,
                 'mAP@0.5':      f"{m.get('map50',     0):.4f}",
                 'mAP@0.5:0.95': f"{m.get('map50_95',  0):.4f}",
                 'Precision':    f"{m.get('precision',  0):.4f}",
                 'Recall':       f"{m.get('recall',     0):.4f}",
                 'AP@0.5':       '—'})

df_results = pd.DataFrame(rows)
print('\n── Final Results Table ──')
print(df_results.to_string(index=False))

# Save as CSV
csv_out = RESULTS_DIR / 'metrics' / 'final_results_table.csv'
df_results.to_csv(csv_out, index=False)
print(f'\nSaved → {csv_out}')

## 4. Results Bar Chart

In [ ]:
if not df_results.empty:
    df_plot = df_results.copy()
    numeric_cols = ['Precision', 'Recall', 'mAP@0.5']
    for col in numeric_cols:
        df_plot[col] = pd.to_numeric(df_plot[col], errors='coerce')

    x = np.arange(len(df_plot))
    width = 0.25
    fig, ax = plt.subplots(figsize=(max(10, len(df_plot)*2), 5))

    ax.bar(x - width, df_plot['Precision'], width, label='Precision', color='#3498db')
    ax.bar(x,         df_plot['Recall'],    width, label='Recall',    color='#e74c3c')
    ax.bar(x + width, df_plot['mAP@0.5'],  width, label='mAP@0.5',   color='#2ecc71')

    ax.set_xticks(x)
    ax.set_xticklabels(df_plot['Model'], rotation=15, ha='right', fontsize=9)
    ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
    ax.set_title('Model Comparison — Test Set Results', fontsize=13)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    out = RESULTS_DIR / 'figures' / 'final_model_comparison.png'
    plt.savefig(out, dpi=150)
    plt.show()
    print(f'Saved → {out}')

## 5. Qualitative Examples — Successes & Failures

In [ ]:
BEST_CKPT = PROJECT_ROOT / 'results' / 'checkpoints' / 'gtsdb_yolov8n_v1_best.pt'

if not BEST_CKPT.exists():
    print(f'Best checkpoint not found: {BEST_CKPT}. Run notebook 04 first.')
else:
    best_model  = YOLO(str(BEST_CKPT))
    img_paths   = sorted((TEST_DIR / 'images').glob('*.jpg'))
    lbl_dir     = TEST_DIR / 'labels'

    success_imgs, failure_imgs = [], []
    success_titles, failure_titles = [], []

    for img_path in img_paths[:50]:  # sample first 50 test images
        img = cv2.imread(str(img_path))
        if img is None: continue
        h, w = img.shape[:2]

        # Ground truth boxes
        gt_boxes = []
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        xc,yc,bw,bh = map(float, parts[1:])
                        gt_boxes.append([int((xc-bw/2)*w), int((yc-bh/2)*h),
                                         int((xc+bw/2)*w), int((yc+bh/2)*h)])

        results = best_model.predict(img, conf=0.25, iou=0.45, verbose=False)
        annotated = results[0].plot()

        # Draw GT boxes (red dashed)
        for (x1,y1,x2,y2) in gt_boxes:
            cv2.rectangle(annotated, (x1,y1), (x2,y2), (255,0,0), 2)

        pred_boxes = results[0].boxes
        n_pred = len(pred_boxes) if pred_boxes is not None else 0
        n_gt   = len(gt_boxes)

        # Classify as success (all GT matched) or failure (missed or many FP)
        if n_gt > 0 and n_pred == n_gt and len(success_imgs) < 6:
            success_imgs.append(annotated)
            success_titles.append(f'GT:{n_gt} Pred:{n_pred} ✅')
        elif (n_gt > 0 and n_pred == 0) and len(failure_imgs) < 6:
            failure_imgs.append(annotated)
            failure_titles.append(f'GT:{n_gt} Pred:{n_pred} ❌ Missed')

    # Success grid
    if success_imgs:
        save_qualitative_grid(
            success_imgs, success_titles,
            out_path=RESULTS_DIR / 'qualitative_examples' / 'successes.png',
            n_cols=3, img_size=(400, 300)
        )

    # Failure grid
    if failure_imgs:
        save_qualitative_grid(
            failure_imgs, failure_titles,
            out_path=RESULTS_DIR / 'qualitative_examples' / 'failures.png',
            n_cols=3, img_size=(400, 300)
        )

    print(f'Success examples: {len(success_imgs)} | Failure examples: {len(failure_imgs)}')

## 6. Failure Mode Analysis

In [ ]:
# Categorise failures by type
# This cell requires the above evaluation loop to have run

print('Failure Mode Checklist for Final Report')
print('=' * 50)
print()
failure_modes = [
    ('Small / distant signs', 'Signs < 32×32 px — model resolution too low?'),
    ('Motion blur',           'Fast-moving or low-shutter images'),
    ('Night / low light',     'Signs with faded colours in low illumination'),
    ('Occlusion',             'Signs partially hidden by vehicles or vegetation'),
    ('Rare classes',          'Classes with few training examples'),
    ('False positives',       'Round or triangular objects misclassified as signs'),
]
for (mode, description) in failure_modes:
    print(f'  ☐ {mode}')
    print(f'     {description}')
    print()

print('Fill in observed failure counts after reviewing qualitative_examples/failures.png')

## 7. Final Report Notes

Fill in after completing all evaluations:

### Key Numbers for the Report
| Metric | Classical | Zero-Shot | Fine-Tuned (best) |
|---|---|---|---|
| mAP@0.5 | — | — | _____ |
| Precision | _____ | _____ | _____ |
| Recall | _____ | _____ | _____ |
| F1 | _____ | _____ | _____ |

### Ablation Results
| Ablation | mAP@0.5 | Delta |
|---|---|---|
| With augmentation | _____ | baseline |
| Without augmentation | _____ | _____ |
| YOLOv8n (nano) | _____ | — |
| YOLOv8s (small) | _____ | _____ |

### Conclusions
- Goal achieved (minimum / expected / stretch): _____
- Main limitation: _____
- Next step if given more time: _____